In [1]:
# Librerías
import pandas as pd
from pathlib import Path
import ee

/home/ddayann/miniconda3/envs/tf_cuda_gee/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


#### Mecanismo de autorización
1. Al ejecutar las siguientes líneas se habilitará un link de acceso
2. Asegura los permisos apropiados para google Earth Engine y demás necesarios
3. Se genera un Token
4. Copiar y pegar el token en la ventana emergente que aparece en la ventana superior de VSC

* Para habilitar nuevos usuarios es necesario desmarcar "ee.Authenticate(force=True)" - Actualmente ya tiene habilitado el ingreso del usuario presente.  
(4/1AeoWuM9_9T1gCVAvTELP8S2YXcFW26H8_p7npV3xK4wc_OUw6Fb5i68ThS8)

In [2]:
# Log Google Earth Engine
# ee.Authenticate(force=True)
ee.Initialize(project='ee-cafe-494102')

In [3]:
# Cargar asset de municipios
municipios_ee = ee.FeatureCollection("projects/ee-cafe-494102/assets/municipios")

# Validación
print("Número de municipios:", municipios_ee.size().getInfo())
print("Columnas:", municipios_ee.first().propertyNames().getInfo())

Número de municipios: 27
Columnas: ['OBJECTID', 'MpNombre', 'Depto', 'MpCodigo', 'MpCategor', 'SHAPE_Leng', 'MpAltitud', 'SHAPE_Area', 'MpArea', 'system:index', 'MpNorma', 'Restriccio']


In [4]:
# Fechas de análisis
fecha_inicio = "2005-01-01"
fecha_fin = "2026-01-01"

In [14]:
# Cargar ERA5-Land diario
era5_temp = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate(fecha_inicio, fecha_fin)
    .select([
        "temperature_2m",
        "temperature_2m_min",
        "temperature_2m_max"
    ])
)

In [ ]:
def reducir_temperatura_diaria(img):
    fecha = ee.Date(img.get("system:time_start"))
    fecha_str = fecha.format("YYYY-MM-dd")

    # Kelvin → Celsius
    temp_mean = img.select("temperature_2m").subtract(273.15).rename("temp_mean_c")
    temp_min = img.select("temperature_2m_min").subtract(273.15).rename("temp_min_c")
    temp_max = img.select("temperature_2m_max").subtract(273.15).rename("temp_max_c")

    img_c = temp_mean.addBands(temp_min).addBands(temp_max)

    reduccion = img_c.reduceRegions(
        collection=municipios_ee,
        reducer=ee.Reducer.mean()
            .combine(
                reducer2=ee.Reducer.count(),
                sharedInputs=True
            ),
        scale=11132
    )

    def limpiar(feat):
        return ee.Feature(None, {
            "municipio": feat.get("MpNombre"),
            "date": fecha_str,
            "temp_mean": feat.get("temp_mean_c_mean"),
            "temp_min": feat.get("temp_min_c_mean"),
            "temp_max": feat.get("temp_max_c_mean"),
            "n_pixeles": feat.get("temp_mean_c_count")
        })

    return reduccion.map(limpiar)

In [12]:
temperatura_municipal = era5_temp.map(reducir_temperatura_diaria).flatten()

In [ ]:
# Exportar a Google Drive sin geometría
task = ee.batch.Export.table.toDrive(
    collection=temperatura_municipal,
    description="temperatura_municipal_era5_2005_2025",
    folder="GEE_exports",
    fileNamePrefix="temperatura_municipal_era5_2005_2025",
    fileFormat="CSV",
    selectors=[
        "municipio",
        "date",
        "temp_mean",
        "temp_min",
        "temp_max",
        "n_pixeles"
    ]
)
task.start()

print("Exportación iniciada.")
print("Revisa el estado de la tarea en Google Earth Engine Tasks.")

Exportación iniciada.
Revisa el estado de la tarea en Google Earth Engine Tasks.


#### Validación del proceso 
A fin de validar que el proceso se está ejecutando es necesario abrir el proyecto de Google Earth Engine, en la pestaña Task. Es posible ver el avance del proceso.  
Al final del procesamiento el archivo será guardado en Google Drive: "/MyDrive/Gee_exports"

In [34]:
# Lectura archivo CSV exportado
BASE_DIR = Path.cwd().parents[1]
ruta_temperatura = BASE_DIR / "data" / "raw" / "temperatura_municipal_era5_2005_2025.csv"
df_temperatura = pd.read_csv(ruta_temperatura)

df_temperatura

,municipio,date,temp_mean,temp_min,temp_max,n_pixeles
0,Villamaria,2005-01-01,10.216389,6.357406,15.235833,10
1,Risaralda,2005-01-01,17.621836,13.233104,22.392257,4
2,Neira,2005-01-01,15.051558,9.985555,20.898102,9
3,Anserma,2005-01-01,17.601240,13.059324,22.376143,6
4,Manzanares,2005-01-01,16.766917,13.606304,21.446842,4
...,...,...,...,...,...,...
207085,Belalcázar,2025-12-31,18.603498,14.373087,23.795296,6
207086,San José,2025-12-31,18.318545,13.928119,22.609562,4
207087,Marquetalia,2025-12-31,18.438983,13.659479,23.350233,2
207088,Norcasia,2025-12-31,24.185866,18.934150,29.687788,8


In [35]:
df_temperatura.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207090 entries, 0 to 207089
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   municipio  207090 non-null  object 
 1   date       207090 non-null  object 
 2   temp_mean  207090 non-null  float64
 3   temp_min   207090 non-null  float64
 4   temp_max   207090 non-null  float64
 5   n_pixeles  207090 non-null  int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 9.5+ MB


In [ ]:
# Tipificación
df_temperatura["municipio"] = df_temperatura["municipio"].astype("category")
df_temperatura["date"] = pd.to_datetime(df_temperatura["date"])

In [37]:
df_temperatura.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207090 entries, 0 to 207089
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   municipio  207090 non-null  category      
 1   date       207090 non-null  datetime64[ns]
 2   temp_mean  207090 non-null  float64       
 3   temp_min   207090 non-null  float64       
 4   temp_max   207090 non-null  float64       
 5   n_pixeles  207090 non-null  int64         
dtypes: category(1), datetime64[ns](1), float64(3), int64(1)
memory usage: 8.1 MB


In [38]:
# Agregación mensual
df_temperatura["mes"] = df_temperatura["date"].dt.month
df_temperatura["anio"] = df_temperatura["date"].dt.year

# Agregación mensual
temperatura_mes = (
    df_temperatura
    .groupby(["municipio", "anio", "mes"], observed=False)
    .agg(
        temp_mean=("temp_mean", "mean"),              # temperatura promedio del mes
        temp_max=("temp_max", "max"),                 # mayor temperatura del mes
        temp_min=("temp_min", "min"),                 # menor temperatura del mes
        n_pixeles=("n_pixeles", "mean")
    )
    .reset_index()
)

# Crear fecha base: primer día del mes
temperatura_mes["date"] = pd.to_datetime(
    temperatura_mes["anio"].astype(str) + "-" +
    temperatura_mes["mes"].astype(str) + "-01"
)

# Llevar al último día del mes
temperatura_mes["date"] = (
    temperatura_mes["date"] + pd.offsets.MonthEnd(0)
)

# Ordenar columnas
temperatura_mes = temperatura_mes[
    [
        "municipio",
        "date",
        "anio",
        "mes",
        "temp_mean",
        "temp_min",
        "temp_max",
        "n_pixeles"
    ]
]

In [45]:
temperatura_anual = (
    temperatura_mes
    .groupby(["municipio", "anio"], observed=False)
    .agg(
        temp_mean=("temp_mean", "mean"),              # temperatura promedio del año
        temp_min=("temp_min", "min"),                 # menor temperatura del año
        temp_max=("temp_max", "max"),                 # mayor temperatura del año
        n_pixeles=("n_pixeles", "mean")
    )
    .reset_index()
)
# Fecha a cierre del año
temperatura_anual["date"] = pd.to_datetime(
    temperatura_anual["anio"].astype(str) + "-12-31"
)
temperatura_anual["mes"] = 12
temperatura_anual = temperatura_anual[
    [
        "municipio",
        "date",
        "anio",
        "temp_mean",
        "temp_min",
        "temp_max",
        "n_pixeles"
    ]
]

In [46]:
display(temperatura_mes.head(12))
display(temperatura_anual.head(21))

,municipio,date,anio,mes,temp_mean,temp_min,temp_max,n_pixeles
0,Aguadas,2005-01-31,2005,1,17.112281,11.437058,24.170613,11.0
1,Aguadas,2005-02-28,2005,2,18.208079,12.151411,25.367099,11.0
2,Aguadas,2005-03-31,2005,3,18.244485,10.930571,26.278290,11.0
3,Aguadas,2005-04-30,2005,4,17.574295,11.547449,25.285705,11.0
4,Aguadas,2005-05-31,2005,5,17.366461,10.402755,24.666192,11.0
5,Aguadas,2005-06-30,2005,6,17.302496,12.154906,24.941658,11.0
6,Aguadas,2005-07-31,2005,7,17.543301,10.194100,26.111903,11.0
7,Aguadas,2005-08-31,2005,8,17.685419,10.803777,25.733740,11.0
8,Aguadas,2005-09-30,2005,9,18.088053,10.874651,26.198838,11.0
9,Aguadas,2005-10-31,2005,10,16.775787,11.358232,25.245632,11.0


,municipio,date,anio,temp_mean,temp_min,temp_max,n_pixeles
0,Aguadas,2005-12-31,2005,17.463806,10.194100,26.278290,11.0
1,Aguadas,2006-12-31,2006,17.305318,10.457883,27.345317,11.0
2,Aguadas,2007-12-31,2007,17.091233,10.312653,27.030563,11.0
3,Aguadas,2008-12-31,2008,16.696953,10.095356,25.342598,11.0
4,Aguadas,2009-12-31,2009,17.257226,10.502291,27.553357,11.0
5,Aguadas,2010-12-31,2010,17.454951,10.582733,27.844509,11.0
6,Aguadas,2011-12-31,2011,16.848704,9.944584,25.809694,11.0
7,Aguadas,2012-12-31,2012,17.251934,9.952210,27.153049,11.0
8,Aguadas,2013-12-31,2013,17.392709,9.829633,26.791317,11.0
9,Aguadas,2014-12-31,2014,17.696478,10.610506,28.033495,11.0


In [47]:
ruta_salida_mes = BASE_DIR / "data" / "processed" / "temperatura_caldas_mes_2005-2025.csv"
temperatura_mes.to_csv(ruta_salida_mes, index=False)
ruta_salida_ano = BASE_DIR / "data" / "processed" / "temperatura_caldas_anio_2005-2025.csv"
temperatura_anual.to_csv(ruta_salida_ano, index=False)